[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Backup and Copying &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings, in the
default rollback journal mode, as the notebook's Setup did, and defines `check`. Run it first. The
tasks do not depend on one another, since every task that changes a database changes a copy, and the
last cell removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "stations.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
NEW_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany(NEW_READING, ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()


def check(path):
    """Open a database file on its own connection, and return its readings and the first line of its integrity_check."""
    conn = sqlite3.connect(path)
    try:
        integrity = conn.execute("PRAGMA integrity_check").fetchone()[0]
        return conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0], integrity
    finally:
        conn.close()


print("built", DATABASE, "| readings and integrity_check:", check(DATABASE))


built scratch/stations.db | readings and integrity_check: (35040, 'ok')


**1.** A backup in batches of 40 pages.


In [2]:
batches = []


def count_batches(status, remaining, total):
    """Called by backup after every batch, with the pages left and the pages in all."""
    batches.append((remaining, total))


source = sqlite3.connect(DATABASE)
target = sqlite3.connect(SCRATCH / "in_batches.db")
source.backup(target, pages=40, progress=count_batches)
target.close()
source.close()

remaining, total = batches[-1]
print(f"{len(batches)} batches for {total} pages, with {remaining} left after the last")
print("the copy:", check(SCRATCH / "in_batches.db"))


8 batches for 301 pages, with 0 left after the last
the copy: (35040, 'ok')


`progress` was called after every batch, so the last call reported no pages left. Nothing wrote to
`stations.db` meanwhile, so no batch had to start over, and there were as many batches as it takes
to copy the pages 40 at a time.


**2.** The same name twice.


In [3]:
conn = sqlite3.connect(DATABASE, autocommit=True)
name = SCRATCH / "stations-2026-02-01.db"
conn.execute("VACUUM INTO ?", (str(name),))
try:
    conn.execute("VACUUM INTO ?", (str(name),))
except sqlite3.OperationalError as error:
    if str(error) != "output file already exists":
        raise
    print("refused:", error)
    name = name.with_stem(name.stem + "-2")
    conn.execute("VACUUM INTO ?", (str(name),))
conn.close()

print(sorted(path.name for path in SCRATCH.glob("stations-2026-*")))
print(name.name, check(name))


refused: output file already exists
['stations-2026-02-01-2.db', 'stations-2026-02-01.db']
stations-2026-02-01-2.db (35040, 'ok')


The second `VACUUM INTO` raised, and left the first copy as it was. The `except` handles only that
refusal and raises any other error again, since a full disk, say, is not solved by a new name.
`with_stem` changes a path's name and keeps its `.db` suffix.


**3.** A dump, restored, with the same temperatures.


In [4]:
source = sqlite3.connect(DATABASE, autocommit=True)
source.execute("BEGIN")
dump = "\n".join(source.iterdump())
source.execute("COMMIT")

restored = sqlite3.connect(SCRATCH / "from_the_dump.db")
restored.executescript(dump)

TOTAL = "SELECT ROUND(SUM(celsius), 1), COUNT(celsius) FROM readings"
print("stations.db:     ", source.execute(TOTAL).fetchone())
print("from_the_dump.db:", restored.execute(TOTAL).fetchone())
restored.close()
source.close()


stations.db:      (118524.5, 35016)
from_the_dump.db: (118524.5, 35016)


A dump writes every `REAL` with as many digits as it takes to read back as the same number, so the
restored temperatures are the same numbers, and so is their sum. `ROUND` keeps the sum to the one
decimal place the readings have, and `COUNT(celsius)` counts the readings that are not `NULL`.


**4.** A database in memory, changed on its own.


In [5]:
source = sqlite3.connect(DATABASE)
in_memory = sqlite3.connect(":memory:")
in_memory.deserialize(source.serialize())

SVALBARD = "SELECT COUNT(*) FROM readings WHERE station_id = (SELECT id FROM stations WHERE name = 'Svalbard')"
with in_memory:
    in_memory.execute("DELETE FROM readings WHERE station_id = (SELECT id FROM stations WHERE name = 'Svalbard')")
print("Svalbard's readings in memory:  ", in_memory.execute(SVALBARD).fetchone()[0])
print("Svalbard's readings in the file:", source.execute(SVALBARD).fetchone()[0])
in_memory.close()
source.close()


Svalbard's readings in memory:   0
Svalbard's readings in the file: 8760


`deserialize` gave the connection in memory a database of its own, made from the bytes, so deleting
from it changed nothing in `stations.db`. Loading the same bytes again is a quick way to give every
test a fresh copy of one database.


**5.** Which copy keeps WAL mode.


In [6]:
conn = sqlite3.connect(shutil.copy(DATABASE, SCRATCH / "wal_copy.db"), autocommit=True)
conn.execute("PRAGMA journal_mode = WAL")

by_backup = sqlite3.connect(SCRATCH / "by_backup.db")
conn.backup(by_backup)
by_backup.close()
conn.execute("VACUUM INTO ?", (str(SCRATCH / "by_vacuum_into.db"),))
conn.close()

for name in ("by_backup.db", "by_vacuum_into.db"):
    copy = sqlite3.connect(SCRATCH / name)
    print(f"{name:<18}", copy.execute("PRAGMA journal_mode").fetchone()[0])
    copy.close()


by_backup.db       wal
by_vacuum_into.db  delete


`backup` copies pages, and the first page of a database records its journal mode, so that copy is in
WAL mode, as its source was. `VACUUM INTO` builds a new database, which starts in the rollback
journal mode, so a program that puts that copy back in place has to set WAL mode again.


**6.** A backup restored beside the database in use.


In [7]:
in_use = sqlite3.connect(shutil.copy(DATABASE, SCRATCH / "in_use.db"), autocommit=True)
in_use.execute("VACUUM INTO ?", (str(SCRATCH / "in_use-backup.db"),))
in_use.execute(NEW_READING, (ids["Tromso"], "2026-01-01T00:00", -7.5))

backup = sqlite3.connect(SCRATCH / "in_use-backup.db")
restored = sqlite3.connect(SCRATCH / "restored.db")
backup.backup(restored)
backup.close()

READINGS = "SELECT COUNT(*) FROM readings"
print("the database in use:", in_use.execute(READINGS).fetchone()[0])
print("the restored backup:", restored.execute(READINGS).fetchone()[0])
restored.close()
in_use.close()


the database in use: 35041
the restored backup: 35040


The restored backup is the database as it was before the new reading, and it sits beside the database
in use, which kept that reading. Rows the database in use has lost can be copied back from the
restored file, and nothing written since the backup is at risk.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Backup and Copying](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/18-backup-and-copying.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
